# 04 — Validaciones y Exploración

Validaciones de calidad sobre `NYC_TAXI_P3.ANALYTICS.OBT_TRIPS`:
- Nulos en columnas esenciales
- Rangos lógicos (duración, distancia, montos)
- Coherencia de fechas (pickup < dropoff)
- Conteos por servicio/año/mes
- Detección de outliers

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("=" * 60)
print("CELDA 1: Inicialización Spark + Configuración Snowflake")
print("=" * 60)

spark = SparkSession.builder.appName("NYC_Taxi_Validation").getOrCreate()
print("✓ SparkSession creada")

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}
SF_OPTS_ANALYTICS = {**SF_OPTIONS, "sfSchema": os.environ["SF_SCHEMA_ANALYTICS"]}

def query_obt(sql):
    return spark.read.format("snowflake").options(**SF_OPTS_ANALYTICS).option("query", sql).load()

print(f"✓ Snowflake URL: {SF_OPTIONS['sfURL']}")
print(f"✓ Schema ANALYTICS: {os.environ['SF_SCHEMA_ANALYTICS']}")
print("✓ Listo para validaciones")
print("=" * 60)

## 4.1 Resumen general de la OBT

In [ ]:
print("=" * 60)
print("CELDA 2: Resumen general de la OBT")
print("=" * 60)

summary_sql = """
SELECT
    COUNT(*)                          AS total_rows,
    COUNT(DISTINCT service_type)      AS distinct_services,
    COUNT(DISTINCT year)              AS distinct_years,
    MIN(pickup_datetime)              AS min_pickup,
    MAX(pickup_datetime)              AS max_pickup,
    MIN(year)                         AS min_year,
    MAX(year)                         AS max_year
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
print(">>> Consultando resumen...")
query_obt(summary_sql).show(truncate=False)
print("✓ Resumen obtenido")
print("=" * 60)

## 4.2 Nulos en columnas esenciales

In [ ]:
print("=" * 60)
print("CELDA 3: Nulos en columnas esenciales")
print("=" * 60)

nulls_sql = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END)  AS null_pickup_dt,
    SUM(CASE WHEN dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS null_dropoff_dt,
    SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END)   AS null_pu_location,
    SUM(CASE WHEN do_location_id IS NULL THEN 1 ELSE 0 END)   AS null_do_location,
    SUM(CASE WHEN service_type IS NULL THEN 1 ELSE 0 END)     AS null_service_type,
    SUM(CASE WHEN vendor_id IS NULL THEN 1 ELSE 0 END)        AS null_vendor_id,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END)     AS null_total_amount,
    SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END)      AS null_fare_amount,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END)    AS null_trip_distance,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END)  AS null_passenger_count,
    SUM(CASE WHEN run_id IS NULL THEN 1 ELSE 0 END)           AS null_run_id
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
print(">>> Verificando nulos...")
query_obt(nulls_sql).show(truncate=False)
print("✓ Análisis de nulos completado")
print("=" * 60)

## 4.3 Rangos lógicos

Verificamos que duración, distancia y montos estén en rangos razonables.

In [ ]:
print("=" * 60)
print("CELDA 4: Rangos lógicos (duración, distancia, montos)")
print("=" * 60)

ranges_sql = """
SELECT
    COUNT(*) AS total_rows,
    -- Duración
    SUM(CASE WHEN trip_duration_min < 0 THEN 1 ELSE 0 END)      AS negative_duration,
    SUM(CASE WHEN trip_duration_min > 1440 THEN 1 ELSE 0 END)    AS duration_over_24h,
    MIN(trip_duration_min) AS min_duration, MAX(trip_duration_min) AS max_duration,
    -- Distancia
    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END)          AS negative_distance,
    SUM(CASE WHEN trip_distance > 500 THEN 1 ELSE 0 END)        AS distance_over_500mi,
    MIN(trip_distance) AS min_distance, MAX(trip_distance) AS max_distance,
    -- Montos
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END)           AS negative_total,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END)            AS negative_fare,
    SUM(CASE WHEN total_amount > 1000 THEN 1 ELSE 0 END)        AS total_over_1000,
    MIN(total_amount) AS min_total, MAX(total_amount) AS max_total,
    -- Velocidad
    SUM(CASE WHEN avg_speed_mph > 100 THEN 1 ELSE 0 END)        AS speed_over_100mph,
    -- Propina
    SUM(CASE WHEN tip_pct > 100 THEN 1 ELSE 0 END)              AS tip_over_100pct
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
print(">>> Verificando rangos...")
query_obt(ranges_sql).show(truncate=False)
print("✓ Rangos verificados")
print("=" * 60)

## 4.4 Coherencia de fechas (pickup < dropoff)

In [ ]:
print("=" * 60)
print("CELDA 5: Coherencia de fechas (pickup < dropoff)")
print("=" * 60)

date_coherence_sql = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN dropoff_datetime < pickup_datetime THEN 1 ELSE 0 END) AS dropoff_before_pickup,
    SUM(CASE WHEN pickup_datetime = dropoff_datetime THEN 1 ELSE 0 END) AS same_pickup_dropoff,
    SUM(CASE WHEN pickup_date <> pickup_datetime::DATE THEN 1 ELSE 0 END) AS pickup_date_mismatch,
    SUM(CASE WHEN EXTRACT(YEAR FROM pickup_datetime) < 2010
             OR EXTRACT(YEAR FROM pickup_datetime) > 2026 THEN 1 ELSE 0 END) AS pickup_year_out_of_range,
    SUM(CASE WHEN EXTRACT(YEAR FROM dropoff_datetime) < 2010
             OR EXTRACT(YEAR FROM dropoff_datetime) > 2026 THEN 1 ELSE 0 END) AS dropoff_year_out_of_range
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pickup_datetime IS NOT NULL AND dropoff_datetime IS NOT NULL
"""
print(">>> Verificando coherencia de fechas...")
query_obt(date_coherence_sql).show(truncate=False)
print("✓ Coherencia de fechas verificada")
print("=" * 60)

## 4.5 Conteos por servicio / año / mes

Matriz de cobertura para confirmar que todos los meses esperados están presentes.

In [ ]:
print("=" * 60)
print("CELDA 6: Cobertura servicio/año/mes")
print("=" * 60)

coverage_sql = """
SELECT service_type, year, month, COUNT(*) AS trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
GROUP BY service_type, year, month
ORDER BY service_type, year, month
"""
print(">>> Consultando cobertura...")
df_coverage = query_obt(coverage_sql)
print(f"✓ Combinaciones servicio/año/mes: {df_coverage.count()}")
df_coverage.show(200, truncate=False)
print("=" * 60)

In [ ]:
print("=" * 60)
print("CELDA 7: Comparar con INGESTION_LOG")
print("=" * 60)

log_sql = """
SELECT service_type, source_year, source_month, rows_loaded, status
FROM NYC_TAXI_P3.RAW.INGESTION_LOG
ORDER BY service_type, source_year, source_month
"""
print(">>> Consultando INGESTION_LOG...")
df_log = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", log_sql).load())
print(f"✓ Registros en INGESTION_LOG: {df_log.count()}")
df_log.show(200, truncate=False)
print("=" * 60)

## 4.6 Detección de outliers extremos

In [ ]:
print("=" * 60)
print("CELDA 8: Detección de outliers extremos")
print("=" * 60)

outlier_sql = """
SELECT
    'trip_duration_min > 1440 (24h)' AS rule,
    COUNT(*) AS flagged_rows
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE trip_duration_min > 1440
UNION ALL
SELECT 'trip_distance > 500 mi', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE trip_distance > 500
UNION ALL
SELECT 'total_amount > $1000', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE total_amount > 1000
UNION ALL
SELECT 'avg_speed_mph > 100', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE avg_speed_mph > 100
UNION ALL
SELECT 'tip_pct > 100%', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE tip_pct > 100
UNION ALL
SELECT 'negative total_amount', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE total_amount < 0
UNION ALL
SELECT 'dropoff < pickup', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE dropoff_datetime < pickup_datetime
UNION ALL
SELECT 'passenger_count = 0', COUNT(*)
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS WHERE passenger_count = 0
"""
print(">>> Detectando outliers...")
query_obt(outlier_sql).show(truncate=False)
print("✓ Outliers detectados")
print("=" * 60)

## 4.7 Resumen de calidad

Consolidado final con métricas de calidad.

In [ ]:
print("=" * 60)
print("CELDA 9: Resumen de calidad consolidado")
print("=" * 60)

quality_sql = """
SELECT
    COUNT(*)                                             AS total_rows,
    COUNT(DISTINCT service_type)                         AS distinct_services,
    MIN(pickup_datetime)                                 AS min_pickup,
    MAX(pickup_datetime)                                 AS max_pickup,
    ROUND(AVG(trip_duration_min), 2)                     AS avg_duration_min,
    ROUND(AVG(trip_distance), 2)                         AS avg_distance_mi,
    ROUND(AVG(total_amount), 2)                          AS avg_total_amount,
    ROUND(AVG(tip_pct), 2)                               AS avg_tip_pct,
    ROUND(AVG(avg_speed_mph), 2)                         AS avg_speed,
    SUM(CASE WHEN trip_duration_min < 0 THEN 1 ELSE 0 END)    AS negative_duration,
    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END)        AS negative_distance,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END)         AS negative_amount,
    SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END)   AS null_pu_location,
    SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END)  AS null_pickup_dt
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
"""
print(">>> Consultando resumen de calidad...")
query_obt(quality_sql).show(truncate=False)
print("✓ NB04 completado — Validaciones y Exploración")
print("=" * 60)